# 03 — Model Training, Registration & Inference

1. Build a training dataset using **`FeatureLookup`** — joins labels with features from the Feature Table
2. Train a model using an **sklearn Pipeline** (ColumnTransformer + LightGBM)
3. Log with **`fe.log_model()`** — captures feature lineage in Unity Catalog
4. Set the **production** alias
5. Run batch inference with **`fe.score_batch()`** on held-out data

In [ ]:
%pip install databricks-feature-engineering lightgbm
dbutils.library.restartPython()

In [ ]:
%run ./00_Config

In [ ]:
import mlflow
from mlflow import MlflowClient
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, classification_report
import lightgbm as lgb
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

fe = FeatureEngineeringClient(model_registry_uri="databricks-uc")
mlflow_client = MlflowClient()
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.set_registry_uri("databricks-uc")

## 1. Split Data by Time: Training vs Test

Split based on time — train on older data, test on the most recent.

In [ ]:
ground_truth_df = spark.table(RAW_SENSOR_TABLE).select("test_run_id", "timestamp", "cycle_status")

cutoff = ground_truth_df.selectExpr("percentile_approx(timestamp, 0.8)").first()[0]

training_labels_df = ground_truth_df.where(F.col("timestamp") < cutoff)
test_labels_df = ground_truth_df.where(F.col("timestamp") >= cutoff)

print(f"Time cutoff: {cutoff}")
print(f"Training:  {training_labels_df.count():,} rows")
print(f"Test:      {test_labels_df.count():,} rows")

## 2. Build Training Set via FeatureLookup

In [ ]:
model_feature_lookups = [
    FeatureLookup(
        table_name=FEATURE_TABLE,
        lookup_key=["test_run_id"],
        timestamp_lookup_key="timestamp",
    )
]

training_set = fe.create_training_set(
    df=training_labels_df,
    feature_lookups=model_feature_lookups,
    exclude_columns=["test_run_id", "timestamp", "testbench_id"],
    label="cycle_status",
)

training_pdf = training_set.load_df().toPandas()
display(training_pdf.head())

## 3. Train/Validation Split

In [ ]:
feature_cols = [c for c in training_pdf.columns if c != "cycle_status"]

X = training_pdf[feature_cols]
y = training_pdf["cycle_status"].values.ravel()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train: {X_train.shape[0]:,} rows")
print(f"Val:   {X_val.shape[0]:,} rows")
print(f"Class balance (train): {y_train.mean():.2%} positive (cycle)")

## 4. Train LightGBM & Register Model

In [ ]:
mlflow.sklearn.autolog(log_input_examples=True, silent=True)
dataset = mlflow.data.from_pandas(X_train)

lgb_params = {
    "objective": "binary",
    "n_estimators": 300,
    "max_depth": 8,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 50,
    "class_weight": "balanced",
    "random_state": 42,
    "verbose": -1,
}

numerical_transformer = Pipeline(steps=[
    ("converter", FunctionTransformer(lambda df: df.apply(pd.to_numeric, errors="coerce"))),
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    [("numerical", numerical_transformer, feature_cols)],
    remainder="passthrough",
    sparse_threshold=0,
)

with mlflow.start_run(run_name="lightGBM") as run:
    mlflow.log_input(dataset, "training")
    mlflow.log_params(lgb_params)

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", lgb.LGBMClassifier(**lgb_params)),
    ])
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]

    metrics = {
        "val_f1": f1_score(y_val, y_pred),
        "val_precision": precision_score(y_val, y_pred),
        "val_recall": recall_score(y_val, y_pred),
        "val_roc_auc": roc_auc_score(y_val, y_proba),
    }
    mlflow.log_metrics(metrics)

    fe.log_model(
        model=model,
        artifact_path="model",
        flavor=mlflow.sklearn,
        training_set=training_set,
        registered_model_name=MODEL_NAME,
    )

print(f"\nRun ID: {run.info.run_id}")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")
print("\n" + classification_report(y_val, y_pred, target_names=["not_cycle", "cycle"]))

## 5. Set Production Alias

In [ ]:
all_versions = mlflow_client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max([int(v.version) for v in all_versions])

production_alias = "production"
mlflow_client.set_registered_model_alias(MODEL_NAME, production_alias, version=latest_version)

print(f"Model '{MODEL_NAME}' version {latest_version} -> alias '{production_alias}'")

## 6. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importance = pd.Series(
    model.named_steps["classifier"].feature_importances_,
    index=feature_cols,
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
importance.plot.barh(ax=ax, color="#1f77b4")
ax.set_title("LightGBM Feature Importance — Cycle Detection")
ax.set_xlabel("Importance (split count)")
plt.tight_layout()
plt.show()

## 7. Batch Inference on Test Data

Score the test set — the most recent 20% of data by time, never seen during training.

In [ ]:
batch_scoring = test_labels_df.select("test_run_id", "timestamp", "cycle_status")
scored_df = fe.score_batch(model_uri=f"models:/{MODEL_NAME}@{production_alias}", df=batch_scoring, result_type="string")
display(scored_df)

# Evaluate on test set
test_pdf = scored_df.select("cycle_status", "prediction").toPandas()
y_true = test_pdf["cycle_status"].astype(int)
y_pred = test_pdf["prediction"].astype(int)

from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
print(f"Test F1:        {f1_score(y_true, y_pred):.4f}")
print(f"Test Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Test Recall:    {recall_score(y_true, y_pred):.4f}")
print("\n" + classification_report(y_true, y_pred, target_names=["not_cycle", "cycle"]))

## 8. Save Full Predictions

Score all data and save — the next notebook uses these to extract cycle boundaries for the Genie Space.

In [ ]:
all_keys = spark.table(RAW_SENSOR_TABLE).select("test_run_id", "timestamp")
all_predictions = fe.score_batch(model_uri=f"models:/{MODEL_NAME}@{production_alias}", df=all_keys, result_type="string")
all_predictions.write.mode("overwrite").saveAsTable(PREDICTIONS_TABLE)
print(f"Predictions saved to {PREDICTIONS_TABLE}")

## Summary

| Step | API |
|------|-----|
| Build training set | `fe.create_training_set()` via `FeatureLookup` |
| Train model | `sklearn.Pipeline` + LightGBM |
| Log with lineage | `fe.log_model()` |
| Set alias | `client.set_registered_model_alias("production")` |
| Batch inference | `fe.score_batch()` on test data |

**Next**: Open `04_Genie_Space_Setup` to extract cycle boundaries and create a natural language explorer.